Analyzes how artificial intelligence is framed in public discourse by examining thematic labels associated with a large collection of AI-related news articles.

Uses natural language processing (NLP) techniques to preprocess and normalize textual category labels, including tokenization, lemmatization, and part-of-speech tagging.

Extracts nouns and noun phrases to represent core thematic concepts such as work, jobs, economy, education, and ethics.

Aggregates these themes over time (by quarter) to identify shifts in dominant topics and patterns of attention.

Compares the relative prominence of economic, educational, and societal themes to understand how discussions of AI evolve.

Provides a data-driven perspective on how AI discourse reflects broader human concerns, positioning AI as a mirror of changing social priorities.

In [ ]:
import pandas as pd
import nltk
import os
from nltk.tokenize import word_tokenize

# 1) 设定一个肯定有写权限的 nltk_data 目录（建议放在家目录）
NLTK_DIR = os.path.expanduser("~/nltk_data")
os.makedirs(NLTK_DIR, exist_ok=True)

# 2) 告诉 NLTK 去这个目录找资源
nltk.data.path.append(NLTK_DIR)

# 3) 下载资源（punkt 是分词；有些版本还需要 punkt_tab）
nltk.download("punkt", download_dir=NLTK_DIR)
nltk.download("punkt_tab", download_dir=NLTK_DIR)  # 保险起见

from nltk.stem import PorterStemmer

# 0) NLTK resources
nltk.download("punkt")

# 1) Load data
df = pd.read_csv("/Users/ziqiwei/Downloads/dataset_A_news_full_10500.csv")

# 2) Keep non-null text
df = df.dropna(subset=["classes_str"]).copy()
texts = df["classes_str"].astype(str)

# 3) Tokenization
df["tokens"] = texts.apply(word_tokenize)

# (optional) lower + remove non-alphabetic tokens (helps quality)
df["tokens_clean"] = df["tokens"].apply(
    lambda toks: [t.lower() for t in toks if t.isalpha()]
)

# 4) Stemming (use clean tokens)
stemmer = PorterStemmer()
df["stems"] = df["tokens_clean"].apply(lambda toks: [stemmer.stem(t) for t in toks])

# 5) Preview
df[["classes_str", "tokens", "tokens_clean", "stems"]].head()




[nltk_data] Downloading package punkt to /Users/ziqiwei/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/ziqiwei/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package punkt to /Users/ziqiwei/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


,classes_str,tokens,tokens_clean,stems
0,Sentiment (Positive / Negative Feelings); Huma...,"[Sentiment, (, Positive, /, Negative, Feelings...","[sentiment, positive, negative, feelings, huma...","[sentiment, posit, neg, feel, human, role, soc..."
1,"Creativity, Expression & Identity; Work, Jobs ...","[Creativity, ,, Expression, &, Identity, ;, Wo...","[creativity, expression, identity, work, jobs,...","[creativ, express, ident, work, job, economi]"
2,"Society, Ethics & Culture","[Society, ,, Ethics, &, Culture]","[society, ethics, culture]","[societi, ethic, cultur]"
3,"Routine, Lifestyle & Behavior","[Routine, ,, Lifestyle, &, Behavior]","[routine, lifestyle, behavior]","[routin, lifestyl, behavior]"
4,"Learning, Knowledge & Education","[Learning, ,, Knowledge, &, Education]","[learning, knowledge, education]","[learn, knowledg, educ]"


In [19]:
#Lemmatization

from nltk.stem import WordNetLemmatizer
import nltk

nltk.download("wordnet")
nltk.download("omw-1.4")

lemmatizer = WordNetLemmatizer()

df["lemmas"] = df["tokens_clean"].apply(
    lambda toks: [lemmatizer.lemmatize(t) for t in toks]
)

df[["tokens_clean", "lemmas"]].head()



[nltk_data] Downloading package wordnet to /Users/ziqiwei/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /Users/ziqiwei/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


,tokens_clean,lemmas
0,"[sentiment, positive, negative, feelings, huma...","[sentiment, positive, negative, feeling, human..."
1,"[creativity, expression, identity, work, jobs,...","[creativity, expression, identity, work, job, ..."
2,"[society, ethics, culture]","[society, ethic, culture]"
3,"[routine, lifestyle, behavior]","[routine, lifestyle, behavior]"
4,"[learning, knowledge, education]","[learning, knowledge, education]"


In [20]:
import nltk

# 下载新版资源（关键）
nltk.download("averaged_perceptron_tagger_eng")

# 显式指定 tagset
df["pos_tags"] = df["lemmas"].apply(
    lambda toks: nltk.pos_tag(toks, tagset=None)
)

df["pos_tags"].head()

#POS Tagging

nltk.download("averaged_perceptron_tagger")

df["pos_tags"] = df["lemmas"].apply(nltk.pos_tag)
df["pos_tags"].head()


[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /Users/ziqiwei/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /Users/ziqiwei/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


0    [(sentiment, NN), (positive, JJ), (negative, J...
1    [(creativity, NN), (expression, NN), (identity...
2          [(society, NN), (ethic, JJ), (culture, NN)]
3     [(routine, JJ), (lifestyle, NN), (behavior, NN)]
4    [(learning, VBG), (knowledge, NN), (education,...
Name: pos_tags, dtype: object

The textual labels were preprocessed using a standard NLP pipeline. We first tokenized the raw text into word-level units, followed by lowercasing and removal of non-alphabetic tokens. We then applied stemming to explore morphological reduction effects, while retaining cleaned tokens for linguistically interpretable analyses.

POS tagging + nouns

In [23]:
import pandas as pd
import spacy
import sys
print(sys.executable)

sys.executable
from collections import Counter

# 0) Load spaCy model FIRST
nlp = spacy.load("en_core_web_sm")

# 1) Clean text column ONCE
df = df.dropna(subset=["classes_str"]).copy()
df["classes_str"] = df["classes_str"].astype(str)

# 2) Ensure lemmas exist (fallback to tokens_clean)
if "lemmas" not in df.columns:
    if "tokens_clean" not in df.columns:
        raise ValueError("Need df['tokens_clean'] or df['lemmas'] before POS tagging.")
    df["lemmas"] = df["tokens_clean"]

# 3) POS tagging
def spacy_pos(tokens):
    doc = nlp(" ".join(tokens))
    return [(t.text, t.pos_) for t in doc]

df["pos_tags"] = df["lemmas"].apply(spacy_pos)

# 4) Extract nouns (theme words)
df["nouns"] = df["pos_tags"].apply(
    lambda tags: [w for w, pos in tags if pos in ("NOUN", "PROPN")]
)

# 5) Preview
df[["classes_str", "pos_tags", "nouns"]].head()


/opt/anaconda3/bin/python


,classes_str,pos_tags,nouns
0,Sentiment (Positive / Negative Feelings); Huma...,"[(sentiment, VERB), (positive, ADJ), (negative...","[feeling, role, society, culture]"
1,"Creativity, Expression & Identity; Work, Jobs ...","[(creativity, NOUN), (expression, NOUN), (iden...","[creativity, expression, identity, work, job, ..."
2,"Society, Ethics & Culture","[(society, NOUN), (ethic, ADJ), (culture, NOUN)]","[society, culture]"
3,"Routine, Lifestyle & Behavior","[(routine, ADJ), (lifestyle, NOUN), (behavior,...","[lifestyle, behavior]"
4,"Learning, Knowledge & Education","[(learning, VERB), (knowledge, NOUN), (educati...","[knowledge, education]"


In [22]:
import sys
print(sys.executable)
import sys
!{sys.executable} -m pip install -U spacy
!{sys.executable} -m spacy download en_core_web_sm



/opt/anaconda3/bin/python
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 46.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 651.7/651.7 kB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 737.1/737.1 kB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 43.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16/16 [spacy]m15/16 [spacy]
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 45.7 MB/s eta 0:00:00 0:00:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [24]:
#Chunking（名词短语 / Noun Phrases）
def extract_noun_phrases(text):
    doc = nlp(text)
    # 去掉纯标点短语，统一小写
    return [chunk.text.lower() for chunk in doc.noun_chunks if any(c.isalnum() for c in chunk.text)]

df["noun_phrases"] = df["classes_str"].apply(extract_noun_phrases)

df[["classes_str", "noun_phrases"]].head(10)


,classes_str,noun_phrases
0,Sentiment (Positive / Negative Feelings); Huma...,"[sentiment, (positive / negative feelings, hum..."
1,"Creativity, Expression & Identity; Work, Jobs ...","[creativity, expression, identity, work, jobs,..."
2,"Society, Ethics & Culture","[society, ethics, culture]"
3,"Routine, Lifestyle & Behavior","[routine, lifestyle, behavior]"
4,"Learning, Knowledge & Education","[learning, knowledge, education]"
5,Technology & Interaction,"[technology, interaction]"
6,Sentiment (Positive / Negative Feelings),"[sentiment, (positive / negative feelings]"
7,"Creativity, Expression & Identity","[creativity, expression, identity]"
8,Technology & Interaction,"[technology, interaction]"
9,Technology & Interaction,"[technology, interaction]"


主题统计 1：全局 Top 30 名词短语（你的“dominant themes”来源）

In [25]:
from collections import Counter
import pandas as pd

all_phrases = [p for sub in df["noun_phrases"] for p in sub]
top_phrases_df = pd.DataFrame(
    Counter(all_phrases).most_common(30),
    columns=["noun_phrase", "count"]
)

top_phrases_df


,noun_phrase,count
0,work,2526
1,jobs,2526
2,economy,2526
3,learning,1946
4,knowledge,1946
5,education,1946
6,technology,1733
7,interaction,1733
8,society,1584
9,ethics,1584


主题统计 2：全局 Top 30 名词（更“词级”的主题线索）

In [27]:
all_nouns = [n for sub in df["nouns"] for n in sub]
top_nouns_df = pd.DataFrame(
    Counter(all_nouns).most_common(30),
    columns=["noun", "count"]
)

top_nouns_df


,noun,count
0,work,2526
1,job,2526
2,economy,2526
3,interaction,2463
4,knowledge,1946
5,education,1946
6,technology,1733
7,society,1584
8,culture,1584
9,lifestyle,1479


对比分析：按 source 看每个来源 Top 10 主题短语

In [28]:
def top_phrases_by_group(group_col, topk=10):
    rows = []
    for g, subdf in df.groupby(group_col):
        phrases = [p for lst in subdf["noun_phrases"] for p in lst]
        for phrase, cnt in Counter(phrases).most_common(topk):
            rows.append({group_col: g, "noun_phrase": phrase, "count": cnt})
    return pd.DataFrame(rows)

top_by_source = top_phrases_by_group("source", topk=10)
top_by_source.sort_values(["source", "count"], ascending=[True, False]).head(50)


,source,noun_phrase,count
0,- Center for Democracy and Technology,routine,1
1,- Center for Democracy and Technology,lifestyle,1
2,- Center for Democracy and Technology,behavior,1
3,- NZCTU,work,1
4,- NZCTU,jobs,1
5,- NZCTU,economy,1
6,- NZCTU,society,1
7,- NZCTU,ethics,1
8,- NZCTU,culture,1
9,100 Mile Free Press,technology,1


时间趋势：按 year 或 quarter 看主题变化

In [30]:
top_by_year = top_phrases_by_group("year", topk=10)
top_by_year.sort_values(["year", "count"], ascending=[True, False]).head(50)

top_by_quarter = top_phrases_by_group("quarter", topk=10)
top_by_quarter.sort_values(["quarter", "count"], ascending=[True, False]).head(50)


,quarter,noun_phrase,count
0,2,work,216
1,2,jobs,216
2,2,economy,216
3,2,learning,179
4,2,knowledge,179
5,2,education,179
6,2,technology,148
7,2,interaction,148
8,2,society,132
9,2,ethics,132
